In [1]:
import numpy as np
from cuml.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from cuml.metrics import pairwise_distances
import tqdm

In [2]:
def loss_fn(x):  # Rastrigin, shifted: many local minima, optimum at all ones, loss 0
    return np.squeeze((x - 1)**2, axis=-1)
    # z = x - 1.0
    # return 10 * x.shape[-1] + np.sum(z**2 - 10 * np.cos(2 * np.pi * z), axis=-1)

In [3]:
def get_feature(latent, center):
	sq_distances = pairwise_distances(latent, center, metric="sqeuclidean")
	return np.exp(-sq_distances / latent.shape[1] / 2)

def smooth_model(model, new_model, alpha):
	model.coef_ = alpha * new_model.coef_ + (1 - alpha) * model.coef_
	model.intercept_ = alpha * new_model.intercept_ + (1 - alpha) * model.intercept_

In [119]:
loss_weighted = True
latent_as_neighbor = True
alpha = 0.05
lambda_ridge = 0.2
k = 16
epsilon = 1e-4
n_iters = 200
batch_size = 128
latent_size = 4
feature_size = 32
parameter_size = 1

In [120]:
model = LinearRegression()
new_model = Ridge(alpha=lambda_ridge, fit_intercept=False)
rng = np.random.default_rng(123)

In [121]:
latent = rng.standard_normal((batch_size, latent_size))
center = rng.standard_normal((feature_size, latent_size))
feature = get_feature(latent, center)
parameter = rng.standard_normal((batch_size, parameter_size))
model.fit(feature, parameter)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 32)","[[ 7.04,-17.69, -7.48,..., 9.15, 20.78,-24.33]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](1,)",[-0.2]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,32
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(32)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](32,)","[7.47,6.25,5.73,...,0.02,0.02,0.01]"


In [122]:
for i in tqdm.trange(n_iters):
	latent = rng.standard_normal((batch_size, latent_size))
	feature = get_feature(latent, center)
	parameter = model.predict(feature).reshape(batch_size, -1)
	loss = loss_fn(parameter)
	distances = pairwise_distances(latent if latent_as_neighbor else parameter)
	neighbor_idx = np.argpartition(distances, kth=k - 1)[:, :k]
	best_local_idx = np.argmin(loss[neighbor_idx], axis=1).reshape(-1, 1)
	best_idx = np.take_along_axis(neighbor_idx, best_local_idx, axis=1)[:, 0]
	best_parameter = parameter[best_idx]
	
	if loss_weighted:
		best_loss = loss[best_idx]
		reduction = loss - best_loss
		movement = np.linalg.norm(parameter - best_parameter, axis=1)
		sample_weight = (reduction + epsilon) / (movement + epsilon)
		new_model.fit(feature, best_parameter, sample_weight)
		smooth_model(model, new_model, alpha)
	else:
		new_model.fit(feature, best_parameter)
		smooth_model(model, new_model, alpha)

100%|██████████| 200/200 [00:01<00:00, 137.10it/s]


## Results

In [123]:
print(parameter)

[[1.09109974]
 [1.07704982]
 [0.75925314]
 [0.89211847]
 [0.63036745]
 [0.99456721]
 [1.09671804]
 [1.07040443]
 [0.79692604]
 [1.02591133]
 [0.93947043]
 [1.02346093]
 [1.04883409]
 [1.08897013]
 [0.94409048]
 [0.79774544]
 [1.04401222]
 [0.91137398]
 [0.9810087 ]
 [1.06801609]
 [0.92056695]
 [0.79554576]
 [0.95044673]
 [1.00470031]
 [0.57557582]
 [1.03259494]
 [0.87812166]
 [1.16943035]
 [0.82282843]
 [0.84000308]
 [1.07696872]
 [1.00901565]
 [1.1407157 ]
 [0.88995317]
 [1.00573343]
 [1.15901409]
 [1.10341122]
 [1.03915062]
 [1.11208661]
 [0.80409847]
 [0.91649074]
 [0.8501604 ]
 [1.09938777]
 [1.21209715]
 [0.68848593]
 [1.0483041 ]
 [1.13063998]
 [0.86418178]
 [1.08235841]
 [1.10022177]
 [1.05366054]
 [0.68893316]
 [0.8622748 ]
 [1.07241778]
 [1.21469663]
 [0.74183301]
 [1.15978676]
 [0.91218303]
 [1.00628688]
 [0.93719267]
 [0.83242027]
 [1.07823954]
 [0.79403615]
 [1.02940204]
 [1.04067052]
 [0.9762879 ]
 [1.18589883]
 [1.15570457]
 [0.90752146]
 [0.78945963]
 [1.19885214]
 [0.59

In [107]:
print(loss_fn(parameter))

[1.72052443e-03 1.44746930e-01 1.84235650e-02 7.83801756e-06
 2.68395479e-03 2.13265759e-06 3.31476367e-02 6.09096842e-03
 2.75568276e-04 3.14097501e-02 1.92263974e-02 6.41422859e-02
 3.09716437e-04 1.18189660e-01 1.92434699e-03 9.66729535e-03
 7.42996279e-04 4.12745247e-02 6.47621941e-04 3.01539668e-02
 6.19562692e-02 1.12840740e-02 1.76008740e-02 1.47964313e-02
 2.31532216e-02 7.34478979e-03 4.26774675e-03 6.74526535e-03
 2.56445806e-02 9.94124550e-02 7.98641497e-05 2.60434780e-01
 1.73214125e-03 3.33894842e-02 4.20372385e-03 1.23669099e-02
 4.97883487e-03 1.70874785e-03 1.22185263e-02 3.09344906e-02
 1.00982584e-02 9.57473702e-05 1.00627121e-02 4.76059168e-03
 2.02009466e-02 1.70520113e-02 4.55236283e-03 6.09134201e-03
 1.75584069e-01 2.19088527e-01 4.08509551e-02 2.85104557e-03
 1.32590894e-03 8.26217591e-04 1.04027237e-02 1.76539137e-02
 2.87961292e-03 3.69053456e-03 6.96707476e-03 2.59884249e-03
 5.90575179e-03 2.19217784e-02 6.87310071e-02 3.40680422e-02
 2.53115113e-03 2.333377

In [55]:
print(loss_fn(parameter - 0.000001)[0])
print(loss_fn(parameter)[0])
print(loss_fn(parameter + 0.000001)[0])

0.0012043747785694522
0.0012044441877758364
0.0012045135989822207
